# L78 Proof Replay — Authoritative Persona Registry Rebase v1

This notebook replays the `L78` proof obligations against the current authoritative surfaces, the H2 employee records, and the H3 lemma/invariant contract.


In [ ]:
from collections import Counter
from copy import deepcopy
from pathlib import Path
import json
import yaml

repo = Path.cwd()
ai_team = json.loads((repo / 'registry' / 'ai_team.json').read_text())
persona_map = yaml.safe_load((repo / 'registry' / 'persona_capability_map.yaml').read_text())
persona_registry = yaml.safe_load((repo / 'registry' / 'persona_registry_v2.yaml').read_text())
runtime_registry = yaml.safe_load((repo / 'registry' / 'agents.yaml').read_text())
employee_records = yaml.safe_load((repo / 'registry' / 'authoritative_digital_employee_records_v1.yaml').read_text())['records']
lemma_text = (repo / 'docs' / 'ma' / 'lemmas' / 'L78_m365_authoritative_persona_registry_rebase_v1.md').read_text()
invariant = yaml.safe_load((repo / 'invariants' / 'lemmas' / 'L78_m365_authoritative_persona_registry_rebase_v1.yaml').read_text())


In [ ]:

APPROVAL_PROFILE_BY_RISK = {
    'low': 'low-observe-create',
    'medium': 'medium-operational',
    'high': 'high-impact',
    'critical': 'critical-regulated',
}
PROMOTED_CAPABILITY_SPEC = {'audit-operations': {'capability_families': ['directory audit review',
                                              'sign-in evidence review',
                                              'provisioning audit tracking'],
                      'workload_families': ['Security, compliance, and data governance',
                                            'Identity, directory, and tenant '
                                            'administration',
                                            'Low-code, workflow, and analytics']},
 'calendar-management-agent': {'capability_families': ['calendar triage and scheduling',
                                                       'availability and meeting '
                                                       'coordination',
                                                       'conflict resolution and '
                                                       'reminder management'],
                               'workload_families': ['Mail, calendar, contacts, and '
                                                     'bookings',
                                                     'Collaboration, meetings, and '
                                                     'communities',
                                                     'Tasks, scheduling, and work '
                                                     'management']},
 'client-relationship-agent': {'capability_families': ['client follow-up orchestration',
                                                       'relationship health scoring',
                                                       'feedback and satisfaction '
                                                       'analysis'],
                               'workload_families': ['Mail, calendar, contacts, and '
                                                     'bookings',
                                                     'Knowledge, search, and employee '
                                                     'experience',
                                                     'Low-code, workflow, and '
                                                     'analytics']},
 'compliance-monitoring-agent': {'capability_families': ['compliance assessment '
                                                         'coordination',
                                                         'policy validation',
                                                         'violation reporting and '
                                                         'remediation planning'],
                                 'workload_families': ['Security, compliance, and data '
                                                       'governance',
                                                       'Identity, directory, and '
                                                       'tenant administration',
                                                       'Low-code, workflow, and '
                                                       'analytics']},
 'device-management': {'capability_families': ['managed device inventory',
                                               'device compliance review',
                                               'endpoint action coordination'],
                       'workload_families': ['Devices, endpoint management, and '
                                             'adjacent Windows admin surfaces',
                                             'Security, compliance, and data '
                                             'governance',
                                             'Identity, directory, and tenant '
                                             'administration']},
 'email-processing-agent': {'capability_families': ['mail triage and classification',
                                                    'response drafting and routing',
                                                    'follow-up scheduling'],
                            'workload_families': ['Mail, calendar, contacts, and '
                                                  'bookings',
                                                  'Tasks, scheduling, and work '
                                                  'management',
                                                  'Documents, notes, and workspace '
                                                  'productivity']},
 'financial-operations-agent': {'capability_families': ['invoice processing '
                                                        'coordination',
                                                        'expense and budget control '
                                                        'tracking',
                                                        'forecast update preparation'],
                                'workload_families': ['Low-code, workflow, and '
                                                      'analytics',
                                                      'Documents, notes, and workspace '
                                                      'productivity',
                                                      'Security, compliance, and data '
                                                      'governance']},
 'identity-security': {'capability_families': ['conditional access governance',
                                               'named location governance',
                                               'identity hardening change control'],
                       'workload_families': ['Identity, directory, and tenant '
                                             'administration',
                                             'Security, compliance, and data '
                                             'governance',
                                             'Low-code, workflow, and analytics']},
 'it-operations-manager': {'capability_families': ['infrastructure monitoring',
                                                   'health-check coordination',
                                                   'alert and backup response'],
                           'workload_families': ['Devices, endpoint management, and '
                                                 'adjacent Windows admin surfaces',
                                                 'Security, compliance, and data '
                                                 'governance',
                                                 'Low-code, workflow, and analytics']},
 'knowledge-management-agent': {'capability_families': ['document indexing',
                                                        'search optimization',
                                                        'training and expert routing'],
                                'workload_families': ['Knowledge, search, and employee '
                                                      'experience',
                                                      'Documents, notes, and workspace '
                                                      'productivity',
                                                      'Content, intranet, and files']},
 'platform-manager': {'capability_families': ['client service provisioning',
                                              'platform lifecycle governance',
                                              'tenant status coordination'],
                      'workload_families': ['Low-code, workflow, and analytics',
                                            'Content, intranet, and files',
                                            'Identity, directory, and tenant '
                                            'administration']},
 'project-coordination-agent': {'capability_families': ['plan and task orchestration',
                                                        'deadline tracking',
                                                        'status reporting'],
                                'workload_families': ['Tasks, scheduling, and work '
                                                      'management',
                                                      'Collaboration, meetings, and '
                                                      'communities',
                                                      'Documents, notes, and workspace '
                                                      'productivity']},
 'project-manager': {'capability_families': ['project lifecycle governance',
                                             'milestone status oversight',
                                             'archive and closeout control'],
                     'workload_families': ['Tasks, scheduling, and work management',
                                           'Collaboration, meetings, and communities',
                                           'Low-code, workflow, and analytics']},
 'recruitment-assistance-agent': {'capability_families': ['candidate screening',
                                                          'interview coordination',
                                                          'offer and onboarding '
                                                          'preparation'],
                                  'workload_families': ['Tasks, scheduling, and work '
                                                        'management',
                                                        'Mail, calendar, contacts, and '
                                                        'bookings',
                                                        'Documents, notes, and '
                                                        'workspace productivity']},
 'reports': {'capability_families': ['usage reporting',
                                     'activity analytics',
                                     'kpi reporting packs'],
             'workload_families': ['Low-code, workflow, and analytics',
                                   'Knowledge, search, and employee experience',
                                   'Documents, notes, and workspace productivity']},
 'security-operations': {'capability_families': ['security alert triage',
                                                 'incident investigation',
                                                 'secure score monitoring'],
                         'workload_families': ['Security, compliance, and data '
                                               'governance',
                                               'Low-code, workflow, and analytics',
                                               'Identity, directory, and tenant '
                                               'administration']},
 'service-health': {'capability_families': ['service health monitoring',
                                            'message center tracking',
                                            'escalation coordination'],
                    'workload_families': ['Low-code, workflow, and analytics',
                                          'Knowledge, search, and employee experience',
                                          'Tasks, scheduling, and work management']},
 'teams-manager': {'capability_families': ['teams workspace governance',
                                           'channel lifecycle administration',
                                           'chat and collaboration coordination'],
                   'workload_families': ['Collaboration, meetings, and communities',
                                         'Mail, calendar, contacts, and bookings',
                                         'Tasks, scheduling, and work management']},
 'ucp-administrator': {'capability_families': ['tier administration',
                                               'tenant config governance',
                                               'audit-log review'],
                       'workload_families': ['Identity, directory, and tenant '
                                             'administration',
                                             'Security, compliance, and data '
                                             'governance',
                                             'Low-code, workflow, and analytics']},
 'website-operations-specialist': {'capability_families': ['website deployment '
                                                           'operations',
                                                           'cdn and dns change '
                                                           'coordination',
                                                           'performance optimization '
                                                           'and rollback readiness'],
                                   'workload_families': ['Media, publishing, and '
                                                         'extended communications',
                                                         'Content, intranet, and files',
                                                         'Low-code, workflow, and '
                                                         'analytics']}}
EXPECTED_PROJECTED_COUNTS = {'communication': 4,
 'design': 5,
 'engineering': 8,
 'hr': 2,
 'marketing': 8,
 'operations': 10,
 'product': 3,
 'project-management': 5,
 'studio-operations': 9,
 'testing': 5}


def approval_profile_for_risk(risk_tier):
    return APPROVAL_PROFILE_BY_RISK[str(risk_tier or 'low')]


def approval_owner_for_agent(agent_definition, department):
    for rule in agent_definition.get('approval_rules', []) or []:
        approvers = [str(approver) for approver in (rule.get('approvers') or []) if approver]
        if approvers:
            return approvers[0]
    return f'department-owner:{department}'


def rebase_ai_team(ai_team_payload, employee_records):
    rebased = deepcopy(ai_team_payload)
    departments = rebased['departments']
    for agent_id, record in employee_records.items():
        members = departments.setdefault(record['department'], [])
        assert all(member.get('agent') != agent_id for member in members)
        members.append({
            'agent': agent_id,
            'name': record['display_name'],
            'role': record['title'],
            'manager': record['manager'],
            'escalation_owner': record['escalation_owner'],
        })
    rebased['total_agents'] = sum(len(members) for members in departments.values())
    rebased['last_updated'] = 'H3-NOTEBOOK-PREVIEW'
    return rebased


def team_entries(ai_team_payload):
    entries = {}
    for department, members in ai_team_payload['departments'].items():
        for member in members:
            entries[member['agent']] = {
                'department': department,
                'display_name': member['name'],
                'title': member.get('title') or member.get('role'),
                'manager': member.get('manager') or f'department-lead:{department}',
                'escalation_owner': member.get('escalation_owner') or f'department-lead:{department}',
            }
    return entries


def rebase_persona_capability_map(persona_map_payload, employee_records, runtime_registry):
    rebased = deepcopy(persona_map_payload)
    for agent_id, record in employee_records.items():
        department = record['department']
        personas = rebased['departments'][department]['personas']
        assert agent_id not in personas
        risk_tier = str(runtime_registry['agents'][agent_id].get('risk_tier') or 'low')
        spec = PROMOTED_CAPABILITY_SPEC[agent_id]
        personas[agent_id] = {
            'display_name': record['display_name'],
            'title': record['title'],
            'current_risk_tier': risk_tier,
            'approval_profile': approval_profile_for_risk(risk_tier),
            'coverage_status': 'persona-contract-only',
            'current_action_count': 0,
            'workload_families': list(spec['workload_families']),
            'capability_families': list(spec['capability_families']),
        }

    total_personas = 0
    registry_backed = 0
    contract_only = 0
    for department_payload in rebased['departments'].values():
        for persona in department_payload['personas'].values():
            total_personas += 1
            if persona['coverage_status'] == 'registry-backed':
                registry_backed += 1
            elif persona['coverage_status'] == 'persona-contract-only':
                contract_only += 1

    rebased['summary'] = {
        'total_personas': total_personas,
        'total_departments': len(rebased['departments']),
        'current_registry_backed_personas': registry_backed,
        'persona_contract_only_personas': contract_only,
        'non_authoritative_registry_agents': 0,
        'non_authoritative_registry_departments': [],
    }
    rebased['last_updated'] = 'H3-NOTEBOOK-PREVIEW'
    return rebased


def stage_registry_document(current_registry_payload, rebased_ai_team, rebased_persona_map, runtime_registry):
    staged = deepcopy(current_registry_payload)
    team = team_entries(rebased_ai_team)
    personas = deepcopy(current_registry_payload['personas'])

    for agent_id, team_entry in team.items():
        if agent_id in personas:
            continue
        capability_entry = None
        for department_payload in rebased_persona_map['departments'].values():
            if agent_id in department_payload['personas']:
                capability_entry = department_payload['personas'][agent_id]
                break
        assert capability_entry is not None
        department = team_entry['department']
        agent_definition = runtime_registry['agents'].get(agent_id, {})
        display_name = team_entry['display_name']
        slug = display_name.lower().replace("'", '').replace(' ', '-')
        personas[agent_id] = {
            'persona_id': agent_id,
            'canonical_agent': agent_id,
            'display_name': display_name,
            'slug': slug,
            'department': department,
            'title': team_entry['title'],
            'manager': team_entry['manager'],
            'escalation_owner': team_entry['escalation_owner'],
            'responsibilities': list(capability_entry['capability_families']),
            'allowed_actions': [],
            'allowed_domains': [],
            'approval_owner': approval_owner_for_agent(agent_definition, department),
            'approval_profile': capability_entry['approval_profile'],
            'risk_tier': capability_entry['current_risk_tier'],
            'coverage_status': 'persona-contract-only',
            'status': 'planned',
            'external_presence_policy': 'internal_only',
            'aliases': sorted({agent_id, display_name.lower(), slug}),
            'action_count': 0,
            'workload_families': list(capability_entry['workload_families']),
            'capability_families': list(capability_entry['capability_families']),
        }

    active_personas = sum(1 for persona in personas.values() if persona['status'] == 'active')
    planned_personas = sum(1 for persona in personas.values() if persona['status'] == 'planned')
    registry_backed = sum(
        1 for persona in personas.values() if persona['coverage_status'] == 'registry-backed'
    )
    contract_only = sum(
        1 for persona in personas.values()
        if persona['coverage_status'] == 'persona-contract-only'
    )

    staged['personas'] = dict(sorted(personas.items()))
    staged['summary'] = {
        'total_personas': len(personas),
        'total_departments': len(rebased_ai_team['departments']),
        'active_personas': active_personas,
        'planned_personas': planned_personas,
        'registry_backed_personas': registry_backed,
        'persona_contract_only_personas': contract_only,
    }
    staged['last_updated'] = 'H3-NOTEBOOK-PREVIEW'
    return staged


Prove that the H3 contract mentions the staged totals and the activation-separation rule, then replay the roster and capability-map obligations.


In [ ]:
rebased_ai_team = rebase_ai_team(ai_team, employee_records)
rebased_persona_map = rebase_persona_capability_map(persona_map, employee_records, runtime_registry)
current_planned = sorted(
    persona_id
    for department_payload in persona_map['departments'].values()
    for persona_id, persona in department_payload['personas'].items()
    if persona['coverage_status'] == 'persona-contract-only'
)
projected_planned = sorted(current_planned + sorted(employee_records))

assert '59' in lemma_text
assert '34' in lemma_text
assert '25' in lemma_text
assert 'allowed_actions = []' in lemma_text
assert invariant['invariant_id'] == 'L78'
assert invariant['ci_gate'] == 'authoritative-persona-registry-rebase-v1'
assert len(employee_records) == 20
assert rebased_ai_team['total_agents'] == 59
assert Counter({department: len(members) for department, members in rebased_ai_team['departments'].items()}) == Counter(EXPECTED_PROJECTED_COUNTS)
assert rebased_persona_map['summary']['persona_contract_only_personas'] == len(projected_planned)
assert rebased_persona_map['summary']['current_registry_backed_personas'] == 34
assert rebased_persona_map['summary']['non_authoritative_registry_agents'] == 0


Replay the staged registry proof and expose the projected planned-persona set that H5 must later activate or deliberately leave deferred.


In [ ]:
staged_registry = stage_registry_document(
    persona_registry,
    rebased_ai_team,
    rebased_persona_map,
    runtime_registry,
)

assert staged_registry['summary']['active_personas'] == 34
assert staged_registry['summary']['planned_personas'] == 25
assert staged_registry['summary']['registry_backed_personas'] == 34
assert staged_registry['summary']['persona_contract_only_personas'] == 25
assert projected_planned == sorted(
    persona_id
    for persona_id, persona in staged_registry['personas'].items()
    if persona['status'] == 'planned'
)

proof_preview = {
    'current_planned_personas': current_planned,
    'projected_planned_personas': projected_planned,
    'registry_summary': staged_registry['summary'],
}
proof_preview
